In [1]:
import os
print(os.getcwd())  # 印出目前工作目錄（notebook 所在資料夾）

/Users/xx/Library/CloudStorage/GoogleDrive-qwas65412@gmail.com/我的雲端硬碟/7.碩士班/Bigdata/m/3.關鍵字總計


In [ ]:
from collections import Counter
import pandas as pd
import ast
import os

# 定義五大個人特質分類
PERSONALITY_TRAITS = {
    "溝通協作力": [
        '溝通', '表達', '協調', '團隊合作', '分享', '說明', '交流', '討論', '傾聽', '互動',
        '團隊', '合作', '協作', '表現力', '參與', '配合度', '部門', '關係', '人際', '社交',
        '關心', '支持', '幫助', '友善', '包容', '信任', '尊重', '理解', '同理心', '同理',
        '互信', '互助', '互惠', '互動性', '互動感', '互動度'
    ],

    "主動進取力": [
        '主動', '積極', '熱情', '學習', '嘗試', '開放', '投入', '進取', '自發', '活力',
        '責任感', '驅動力', '精進', '提升', '進修', '成長', '成就感', '挑戰', '熱忱', '自我',
        '自信', '自我要求', '自我管理', '自我激勵', '自我提升', '思維'
    ],

    "問題解決力": [
        '解決', '獨立', '邏輯', '分析', '思考', '判斷', '規劃', '執行', '效率',
        '改善問題', '優化'
    ],

    "創新思維力": [
        '創新', '創意', '創造力', '突破', '改善', '發想', '構思', '創造', '變通', '前瞻',
        '變化', '調整', '創意性', '創新性', '創造性', '獨特性', '獨創性',
        '新穎性', '新鮮感', '新穎', '新鮮', '新意', '新思維', '新觀點', '新觀念', '跳脫'
    ],

    "穩定專注力": [
        '負責', '細心', '認真', '可靠', '誠實', '配合', '準時', '專注', '抗壓', '壓力',
        '適應', '充實', '紀律', '穩定性', '耐心', '耐性', '耐壓', '耐磨', '耐勞',
        '耐久性', '耐用性', '穩健性', '穩定感', '穩定度', '彈性', '靈活', '靈活性', '調適', '應變'
    ]
}
# 建立詞對分類的映射表
word_to_category = {word: category for category, words in PERSONALITY_TRAITS.items() for word in words}

file_path = os.path.join('..','2.斷詞', '104_preprocessed.csv')
df = pd.read_csv(file_path, sep='|')
jobTypes = ['前端工程師', '數據分析師', 'AI工程師', '網路管理工程師', '雲端工程師']
allowedPOS = ['Na', 'Nb', 'Nc','Nv','Vc']

def get_word_counts():
    """分析一般關鍵字"""
    general_counts = {jobType: Counter() for jobType in jobTypes + ['全部']}
    
    for jobType in jobTypes:
        df_group = df[df.jobType == jobType]
        for row in df_group.token_pos:
            for word, pos in ast.literal_eval(row):
                if (len(word) >= 2) and (pos in allowedPOS):
                    general_counts[jobType][word] += 1
                    general_counts['全部'][word] += 1
    
    return {jobType: general_counts[jobType].most_common(100) for jobType in general_counts}

def get_personality_trait_counts():
    """分析個人特質與五大分類
       修改部分：在同一筆職缺中，對alltraits（即 categorized_traits）只累計每個分類一次，
       即使該筆職缺中有多個關鍵字屬於同一分類也只計一次。"""
    trait_counts = {jobType: Counter() for jobType in jobTypes + ['全部']}
    categorized_traits = {jobType: Counter() for jobType in jobTypes + ['全部']}
    
    for jobType in jobTypes:
        df_group = df[df.jobType == jobType]
        for row in df_group.token_pos:
            # 用集合方式收集該筆職缺中出現的個人特質關鍵字（避免重複計算）
            unique_personality = set()
            for word, _ in ast.literal_eval(row):
                if word in word_to_category:
                    unique_personality.add(word)
                    # 對 trait_counts 仍然是逐一計算
                    trait_counts[jobType][word] += 1
                    trait_counts['全部'][word] += 1
            # 使用 unique_personality 來獲取該筆職缺出現的唯一分類
            unique_categories = {word_to_category[word] for word in unique_personality}
            for category in unique_categories:
                categorized_traits[jobType][category] += 1
                categorized_traits['全部'][category] += 1

    return trait_counts, categorized_traits

# 執行分析
general_counts = get_word_counts()
trait_counts, categorized_traits = get_personality_trait_counts()

# 儲存一般關鍵字
pd.DataFrame([(k, v) for k, v in general_counts.items()],
             columns=['jobType', 'keywords']).to_csv('104_general_keywords.csv', index=False)

# 儲存個人特質關鍵字與五大類統計
df_traits = pd.DataFrame([
    (jobType, trait_counts[jobType].most_common(), dict(categorized_traits[jobType]))
    for jobType in trait_counts
], columns=['jobType', 'traits', 'alltraits'])

df_traits.to_csv('104_personality_traits.csv', index=False)